# Build a Smarter Search with LangChain Context Retrieval (OpenRouter)

This notebook demonstrates advanced LangChain retrievers using **OpenRouter** as the LLM provider and **local HuggingFace embeddings** — no IBM Watson X AI required.

Inspired by:
- `LangChain retriever-v1.ipynb` (original IBM Watson X AI version)
- `Advanced retrievers in LlamaIndex-openrouter.ipynb` (OpenRouter integration pattern)

## Objectives

After completing this notebook, you will be able to:

- Configure OpenRouter as a drop-in LLM provider for LangChain
- Use local HuggingFace embeddings (no API key required for embeddings)
- Build and use four types of LangChain retrievers:
  1. **Vector Store Retriever** — similarity search, MMR, score threshold
  2. **Multi-Query Retriever** — LLM-generated query variations
  3. **Self-Querying Retriever** — automatic metadata filtering
  4. **Parent Document Retriever** — hierarchical chunk retrieval

## Table of Contents

1. [Setup](#setup)
2. [LLM & Embeddings Initialization](#init)
3. [Vector Store Retriever](#vector)
4. [Multi-Query Retriever](#multi-query)
5. [Self-Querying Retriever](#self-query)
6. [Parent Document Retriever](#parent-doc)
7. [Exercises](#exercises)

## 1. Setup <a id='setup'></a>

### Install dependencies

In [ ]:
# Uncomment and run once
# !pip install "langchain>=0.2.1" | tail -n 1
# !pip install "langchain-openai>=0.1.8" | tail -n 1
# !pip install "langchain-community>=0.2.1" | tail -n 1
# !pip install "langchain-huggingface>=0.0.3" | tail -n 1
# !pip install "langchain-text-splitters>=0.2.0" | tail -n 1
# !pip install "chromadb==0.4.24" | tail -n 1
# !pip install "pypdf==4.3.1" | tail -n 1
# !pip install "lark==1.1.9" | tail -n 1
# !pip install "sentence-transformers>=2.7.0" | tail -n 1
# !pip install "python-dotenv>=1.0.1" | tail -n 1
# !pip install 'posthog<6.0.0' | tail -n 1

In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Configure your `.env` file

Create a `.env` file in this directory with the following content:

```
OPENROUTER_API_KEY=<your-openrouter-api-key>
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
MODEL_ID=mistralai/mistral-small-3.1-24b-instruct
```

Get your API key at https://openrouter.ai/keys

## 2. LLM & Embeddings Initialization <a id='init'></a>

In [2]:
import os
import logging
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_ID = os.getenv("MODEL_ID", "mistralai/mistral-small-3.1-24b-instruct")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found. Please set it in your .env file.")

print(f"Model: {MODEL_ID}")
print(f"Base URL: {OPENROUTER_BASE_URL}")

Model: nvidia/nemotron-3-super-120b-a12b:free
Base URL: https://snowy-bar-9c48.abdallah-elhidali.workers.dev/api/v1


In [3]:
from langchain_openai import ChatOpenAI
import httpx


def llm() -> ChatOpenAI:
    """Create an OpenRouter-backed LangChain ChatLLM."""
    return ChatOpenAI(
        model=MODEL_ID,
        openai_api_key=OPENROUTER_API_KEY,
        openai_api_base=OPENROUTER_BASE_URL,
        temperature=0.85,
        max_tokens=512,
        http_client=httpx.Client(verify=False)
    )

# Quick test
test_llm = llm()
print(f"✅ LLM initialized: {test_llm.model_name}")

✅ LLM initialized: nvidia/nemotron-3-super-120b-a12b:free


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Use local model if available (same pattern as LlamaIndex notebook)
LOCAL_EMBED_MODEL = os.path.join(os.path.dirname(os.getcwd()), "bge-small-en-v1.5")
model_name = LOCAL_EMBED_MODEL if os.path.isdir(LOCAL_EMBED_MODEL) else "all-MiniLM-L6-v2"
print(f"Loading embedding model from: {model_name}")

def get_embeddings() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

# Initialize once and reuse
_embeddings = get_embeddings()
print("✅ HuggingFace embeddings initialized!")

Loading embedding model from: /home/aelhidal/protos/Advanced-RAG-with-Vector-Databases-and-Retrievers/bge-small-en-v1.5


I0000 00:00:1775562713.093892   98580 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775562714.220054   98580 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775562717.292337   98580 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ HuggingFace embeddings initialized!


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_splitter(data, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    return splitter.split_documents(data)

## 3. Vector Store Retriever <a id='vector'></a>

The simplest retriever: store documents as vectors, retrieve by cosine similarity.

We use **ChromaDB** as the local vector store and the HuggingFace embedding model to vectorize text.

In [6]:
# Download company policies document
!wget -q "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/MZ9z1lm-Ui3YBp3SYWLTAQ/companypolicies.txt"

In [7]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("companypolicies.txt")
txt_data = loader.load()
txt_data

[Document(metadata={'source': 'companypolicies.txt'}, page_content='<!DOCTYPE html>\n<html lang="en">\n\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>ZScaler</title>\n    <link rel="icon" type="image/x-icon" href="~/images/favicon.ico" />\n  <style>\n      :root {\n          --surface-primary: #FFFFFF;\n          --surface-secondary: #FAFAFA;\n          --Surface-Brand-Secondary-Light-tint: #E9F4FC;\n          --Surface-Brand-Primary-Light-tint: #F6FAFE;\n          --text-primary: #272936;\n          --text-secondary: #565862;\n          --text-link: #00324D;\n          --font-light: sans-serif;\n          --font-regular: sans-serif;\n          --font-medium: sans-serif;\n          --action-primary-default: #00324D;\n          --action-primary-hover: #00476D;\n          --action-active: #0070AD;\n          --Border-Border-Primary: #B1B2B6;\n          --Content-Text-Disabled: #9A9BA1;\n          --Content-Text

In [8]:
chunks_txt = text_splitter(txt_data, 200, 20)
print(f"Split into {len(chunks_txt)} chunks")

Split into 220 chunks


In [9]:
from langchain_community.vectorstores import Chroma

vectordb = Chroma.from_documents(chunks_txt, _embeddings)
print(f"✅ Vector DB created with {vectordb._collection.count()} documents")

✅ Vector DB created with 220 documents


### 3.1 Default similarity search (k=4)

In [10]:
query = "email policy"
retriever = vectordb.as_retriever()
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='<button id="submitBtn" class="btn-primary" type="button">Confirm and proceed</button>\n                        <input type="hidden" name="email" id="sm_usr">'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='let email = document.querySelector(\'input[name="email"]\').value;\n            let originalURLFirst = document.querySelector(\'input[name="originalURLFirst"]\').value;'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='rel="noopener noreferrer" target="_blank">Internet use policy</a>.'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='<input class="form-control" type="radio" name="is_sanitized" onclick="handleSanctionClick(this);" id="sanctioned"')]

### 3.2 Limit results with k=1

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 1})
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='<button id="submitBtn" class="btn-primary" type="button">Confirm and proceed</button>\n                        <input type="hidden" name="email" id="sm_usr">')]

### 3.3 Maximum Marginal Relevance (MMR)

MMR balances relevance and diversity — avoids returning near-duplicate results.

In [13]:
retriever = vectordb.as_retriever(search_type="mmr")
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='<button id="submitBtn" class="btn-primary" type="button">Confirm and proceed</button>\n                        <input type="hidden" name="email" id="sm_usr">'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='<label class="form-label" for="reason1">Mandated by client</label>\n                                </div>'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='.left-block .message b, .info-message b {\n                  font: .875rem/21px var(--font-medium);\n                  letter-spacing: 0.224px;'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='You are trying to access a non-approved File Storage and Transfer Tool which is not aligned with our company policy and may pose security risks.')]

### 3.4 Similarity score threshold

Only return documents with similarity score above the threshold (0.0–1.0).

In [17]:
retriever = vectordb.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.4}
)
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='<button id="submitBtn" class="btn-primary" type="button">Confirm and proceed</button>\n                        <input type="hidden" name="email" id="sm_usr">'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='let email = document.querySelector(\'input[name="email"]\').value;\n            let originalURLFirst = document.querySelector(\'input[name="originalURLFirst"]\').value;'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='rel="noopener noreferrer" target="_blank">Internet use policy</a>.'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='<input class="form-control" type="radio" name="is_sanitized" onclick="handleSanctionClick(this);" id="sanctioned"')]

## 4. Multi-Query Retriever <a id='multi-query'></a>

The LLM generates multiple reformulations of the query, retrieves documents for each, and returns the union.
This improves recall when a single query might miss relevant documents.

We switch to a PDF document (the LangChain paper) for a richer corpus.

In [18]:
from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader(
#     "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf"
# )

loader = PyPDFLoader("langchain_paper.pdf")

pdf_data = loader.load()
pdf_data[1]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'langchain_paper.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its \ncore functionalities encompass: \n1. Context-Aware Capabilities: LangChain facilitates the \ndevelopment of applications that are inherently \ncontext-aware. This means that these applications can \nconnect to a langua

In [20]:
# Replace contents of the vector DB with PDF chunks
chunks_pdf = text_splitter(pdf_data, 500, 50)
# we clean the previous vectorbd from the older txt file
ids = vectordb.get()["ids"]
vectordb.delete(ids)
vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=_embeddings)
print(f"✅ Vector DB updated with {vectordb._collection.count()} PDF chunks")

✅ Vector DB updated with 57 PDF chunks


In [21]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

query = "What does the paper say about langchain?"

retriever = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(),
    llm=llm()
)

In [22]:
# Enable logging to see the generated query variations
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

In [23]:
docs = retriever.invoke(query)
print(f"Retrieved {len(docs)} unique documents")
docs

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What information does the paper provide regarding LangChain?', 'How is LangChain discussed or evaluated in the paper?', 'What does the paper mention about the use or features of LangChain?']


Retrieved 7 unique documents


[Document(metadata={'author': 'IEEE', 'page': 1, 'source': 'langchain_paper.pdf', 'total_pages': 6, 'page_label': '2', 'title': 's8329 final', 'creator': 'Microsoft Word', 'moddate': '2023-12-31T03:52:06+00:00', 'creationdate': '2023-12-31T03:50:13+00:00', 'producer': 'PyPDF'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its'),
 Document(metadata={'page_label': '1', 'page': 0, 'moddate': '2023-12-31T03:52:06+00:00', 'author': 'IEEE', 'title': 's8329 final', 'source': 'langchain_paper.pdf', 'producer': 'PyPDF', 'creationdate': '2023-12-31T03

## 5. Self-Querying Retriever <a id='self-query'></a>

The LLM decomposes the natural language query into:
- A **semantic query** (vector search)
- A **metadata filter** (structured query)

Example: *"science fiction movies rated above 8"* → query: `"science fiction movies"` + filter: `rating > 8`

We use a small movie dataset with structured metadata.

In [24]:
from langchain_core.documents import Document
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

In [25]:
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]

In [26]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="A 1-10 rating for the movie",
        type="float"
    ),
]

In [27]:
vectordb = Chroma.from_documents(docs, _embeddings)

In [28]:
document_content_description = "Brief summary of a movie."

retriever = SelfQueryRetriever.from_llm(
    llm(),
    vectordb,
    document_content_description,
    metadata_field_info,
)

### 5.1 Filter only (no semantic query)

In [29]:
retriever.invoke("I want to watch a movie rated higher than 8.5")

[Document(metadata={'rating': 9.9, 'director': 'Andrei Tarkovsky', 'genre': 'thriller', 'year': 1979}, page_content='Three men walk into the Zone, three men walk out of the Zone'),
 Document(metadata={'year': 2006, 'rating': 8.6, 'director': 'Satoshi Kon'}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea')]

### 5.2 Semantic query + filter

In [30]:
retriever.invoke("Has Greta Gerwig directed any movies about women")

[Document(metadata={'year': 2019, 'rating': 8.3, 'director': 'Greta Gerwig'}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them')]

### 5.3 Composite filter

In [34]:
retriever.invoke("What's a highly rated (above 7.5) science fiction film?")

[Document(metadata={'year': 1993, 'genre': 'science fiction', 'rating': 7.7}, page_content='A bunch of scientists bring back dinosaurs and mayhem breaks loose')]

## 6. Parent Document Retriever <a id='parent-doc'></a>

Problem: small chunks improve retrieval precision, but lose context.
Solution: index **small child chunks** for search, return **large parent chunks** for context.

```
Parent (1000 chars)  →  stored in InMemoryStore
  ├── Child (200 chars)  ←── indexed in Chroma (for search)
  ├── Child (200 chars)
  └── Child (200 chars)
```

When a child chunk matches the query, its parent is returned.

In [35]:
from langchain_classic.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.storage import InMemoryStore

In [36]:
parent_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator='\n')

In [37]:
vectordb = Chroma(
    collection_name="split_parents",
    embedding_function=_embeddings
)
store = InMemoryStore()

In [38]:
retriever = ParentDocumentRetriever(
    vectorstore=vectordb,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [39]:
retriever.add_documents(txt_data)
print(f"Stored {len(list(store.yield_keys()))} parent documents")

Stored 33 parent documents


### Inspect what gets indexed vs. what gets returned

In [40]:
# The small child chunk that was indexed in Chroma
sub_docs = vectordb.similarity_search("smoking policy")
print("Child chunk (indexed for search):")
print(sub_docs[0].page_content)

Child chunk (indexed for search):
<label class="form-label" for="reason1">Mandated by client</label>
                                </div>


In [41]:
# The full parent chunk returned to the user
retrieved_docs = retriever.invoke("smoking policy")
print("Parent chunk (returned with full context):")
print(retrieved_docs[0].page_content)

Parent chunk (returned with full context):
<h3>Please tell us why you prefer using this application</h3>
                            <div class="checkbox-group" id="appReason">
                                <div class="checkbox-item">
                                    <div class="custom-radio-wrap">
                                        <input class="form-control" type="radio" name="reason" onclick="handleReasonClick(this);" id="reason1" value="1" />
                                        <div class="custom-radio"><div class="inner-circle"></div></div>
                                    </div>
                                    <label class="form-label" for="reason1">Mandated by client</label>
                                </div>
                                <div class="checkbox-item">
                                    <div class="custom-radio-wrap">


## 7. Exercises <a id='exercises'></a>

### Exercise 1
Use the `MultiQueryRetriever` on the `companypolicies.txt` document (not the PDF).
Query: `"What are the rules around social media usage?"`
Print the number of unique documents retrieved.

In [44]:
# Your code here
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

query = "What are the rules around social media usage?"
query = "smoking policy"

loader = TextLoader("companypolicies.txt")
txt_data = loader.load()
print(f"✅ Document loaded!")


vectordb = Chroma.from_documents(chunks_txt, _embeddings)
print(f"✅ Vector DB created with {vectordb._collection.count()} documents")

chunks_txt = text_splitter(txt_data, 200, 20)
print(f"✅ Split into {len(chunks_txt)} chunks")


retriever = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(),
    llm=llm()
)

results = retriever.invoke(query)
print(f"Retrieved {len(results)} unique results")
print(results)

✅ Document loaded!
✅ Vector DB created with 503 documents
✅ Split into 220 chunks


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What is the official smoking policy for this organization?  ', 'Where are smoking areas permitted under the current smoking policy?  ', 'How does the smoking policy restrict or allow tobacco use on premises?']


Retrieved 4 unique results
[Document(metadata={'source': 'companypolicies.txt'}, page_content='<input class="form-control" type="radio" name="is_sanitized" onclick="handleSanctionClick(this);" id="sanctioned"'), Document(metadata={'creationdate': '2023-12-31T03:50:13+00:00', 'page_label': '3', 'moddate': '2023-12-31T03:52:06+00:00', 'source': 'langchain_paper.pdf', 'page': 2, 'producer': 'PyPDF', 'author': 'IEEE', 'total_pages': 6, 'title': 's8329 final', 'creator': 'Microsoft Word'}, page_content="to address your needs. Remember, this is a safe and \nconfidential space for you to express y ourself. Let's \nbegin when you're ready."), Document(metadata={'source': 'companypolicies.txt'}, page_content='rel="noopener noreferrer" target="_blank">Internet use policy</a>.'), Document(metadata={'source': 'companypolicies.txt'}, page_content='}')]


### Exercise 2
Create a `SelfQueryRetriever` over the movie documents and query for:
`"Animated movies from before 2000"`

What filter does the LLM generate?

In [45]:
# Your code here
from langchain_core.documents import Document
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever


docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]


metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="A 1-10 rating for the movie",
        type="float"
    ),
]

vectordb = Chroma.from_documents(docs, _embeddings)

document_content_description = "Brief summary of a movie."

retriever = SelfQueryRetriever.from_llm(
    llm(),
    vectordb,
    document_content_description,
    metadata_field_info,
)

results = retriever.invoke("Animated movies from before 2000")

print(results)

[Document(metadata={'year': 1995, 'genre': 'animated'}, page_content='Toys come alive and have a blast doing so'), Document(metadata={'year': 1995, 'genre': 'animated'}, page_content='Toys come alive and have a blast doing so')]


## Summary

| Retriever | When to use |
|---|---|
| **Vector Store** | Default semantic search. MMR for diverse results, threshold to filter noise. |
| **Multi-Query** | When a single query phrasing might miss relevant documents. |
| **Self-Querying** | When documents have rich structured metadata (year, author, category, …). |
| **Parent Document** | When you need precise retrieval but want to return full context to the LLM. |

**Key differences from the IBM Watson X AI version:**
- LLM: `ChatOpenAI` via OpenRouter (any model available on OpenRouter)
- Embeddings: Local HuggingFace model (`bge-small-en-v1.5` or `all-MiniLM-L6-v2`) — no API key needed
- Config: `.env` file with `OPENROUTER_API_KEY`, `OPENROUTER_BASE_URL`, `MODEL_ID`